# M1 Semantic Memory：用户事实与偏好

只增加 semantic memory，保持数据、路由策略、回复模板和评测方式不变。

本轮只回答一个问题：保存用户签名后，应回复邮件的写作合规是否改善？

In [ ]:
import json

from email_memory_lab import (
    AgentConfig,
    EmailAgent,
    LocalStore,
    evaluate_config,
    find_project_root,
    load_labeled_emails,
    seed_memories,
    start_run,
)

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"
cases = load_labeled_emails(DATA / "emails.json")
store = LocalStore()
seed_memories(store, DATA / "memories.json", "user-a")
run = start_run(PROJECT_ROOT, "02-semantic", {"memory": "semantic"})

baseline = evaluate_config(cases, store, "user-a", AgentConfig())
semantic = evaluate_config(cases, store, "user-a", AgentConfig(use_semantic=True))
print(f"Run: {run.run_id}")

## 对照结果

semantic memory 不应改变路由准确率；它改善的是应回复邮件的签名合规。

In [ ]:
for label, result in (("baseline", baseline), ("semantic", semantic)):
    print(label, json.dumps(result["summary"], ensure_ascii=False))
    print("failures:", [row["email_id"] for row in result["failures"]])

In [ ]:
case = next(case for case in cases if case.email.id == "docs-question")
result = EmailAgent(store, "user-a", AgentConfig(use_semantic=True)).handle(case.email)

print(result.response)
print("used_memory:", result.used_memory)
print("context:")
print(result.context)

## 检查点

- 写作合规改善是否来自 semantic memory，而不是路由变化？
- 如果记忆写入成功但没有召回，这个 Notebook 会看到什么？
- 为什么仅凭一个样本不能证明 semantic memory 普遍有效？

In [ ]:
path = run.save(
    "semantic-comparison.json",
    {"baseline": baseline, "semantic": semantic},
)
print(f"Saved: {path}")